### Goal
Convert:

Into:

This makes the system look much closer to ChatGPT.

### import libraries

In [1]:
import pickle
import faiss

### Load Metadata

In [2]:
with open("data/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print("Metadata Loaded:", len(metadata))

Metadata Loaded: 6


### Load Vectorizer

In [3]:
with open("data/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print("Vectorizer Loaded Successfully")

Vectorizer Loaded Successfully


### Load FAISS Index

In [4]:
index = faiss.read_index("data/faiss_index.bin")

print("Vectors Stored:", index.ntotal)

Vectors Stored: 6


### Initialize Chat History

In [25]:
chat_history = []

### Define Retrieval Function

In [26]:
def retrieve_context(question, k=3):

    query_vector = vectorizer.transform([question])
    query_vector = query_vector.toarray().astype("float32")

    distances, indices = index.search(query_vector, k)

    results = []

    for idx in indices[0]:

        results.append({
            "source":metadata[idx]["source"],
            "text":metadata[idx]["text"]
        })

    return results

### Define Chat Function

In [27]:
def chat(question):

    results = retrieve_context(question)

    response = {
        "question": question,
        "answer": results[0]["text"],
        "sources": [results[0]["source"]]
    }

    chat_history.append(response)

    return response

### Ask First Question

In [28]:
response = chat("How many leave days are allowed?")

print(response)

{'question': 'How many leave days are allowed?', 'answer': 'Employee Handbook\n\nEmployees receive 20 annual leave days.\n\nRemote work is allowed three days per week.\n\nWorking hours are 9 AM to 6 PM.\n\nHealth insurance benefits are provided.', 'sources': ['employee_handbook.txt']}


### Ask Second Question

In [29]:
response = chat("What monitoring tools are used?")

print(response)

{'question': 'What monitoring tools are used?', 'answer': 'Cloud Migration Guide\n\nAWS is the primary cloud platform.\n\nKubernetes is used for deployment.\n\nGitHub Actions manages CI/CD.\n\nPrometheus and Grafana are used for monitoring.', 'sources': ['cloud_migration_guide.txt']}


### View Conversation History

In [30]:
for item in chat_history:

    print("=" * 80)

    print("Question:")
    print(item["question"])

    print("\nAnswer:")
    print(item["answer"])

    print("\nSources:")
    print(item["sources"])

Question:
How many leave days are allowed?

Answer:
Employee Handbook

Employees receive 20 annual leave days.

Remote work is allowed three days per week.

Working hours are 9 AM to 6 PM.

Health insurance benefits are provided.

Sources:
['employee_handbook.txt']
Question:
What monitoring tools are used?

Answer:
Cloud Migration Guide

AWS is the primary cloud platform.

Kubernetes is used for deployment.

GitHub Actions manages CI/CD.

Prometheus and Grafana are used for monitoring.

Sources:
['cloud_migration_guide.txt']


In [31]:
print(chat_history)

[{'question': 'How many leave days are allowed?', 'answer': 'Employee Handbook\n\nEmployees receive 20 annual leave days.\n\nRemote work is allowed three days per week.\n\nWorking hours are 9 AM to 6 PM.\n\nHealth insurance benefits are provided.', 'sources': ['employee_handbook.txt']}, {'question': 'What monitoring tools are used?', 'answer': 'Cloud Migration Guide\n\nAWS is the primary cloud platform.\n\nKubernetes is used for deployment.\n\nGitHub Actions manages CI/CD.\n\nPrometheus and Grafana are used for monitoring.', 'sources': ['cloud_migration_guide.txt']}]
